In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('./data/fraudTest.csv', index_col=0)

In [4]:
df.head(2)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 555719 entries, 0 to 555718
Data columns (total 22 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   trans_date_trans_time  555719 non-null  object 
 1   cc_num                 555719 non-null  int64  
 2   merchant               555719 non-null  object 
 3   category               555719 non-null  object 
 4   amt                    555719 non-null  float64
 5   first                  555719 non-null  object 
 6   last                   555719 non-null  object 
 7   gender                 555719 non-null  object 
 8   street                 555719 non-null  object 
 9   city                   555719 non-null  object 
 10  state                  555719 non-null  object 
 11  zip                    555719 non-null  int64  
 12  lat                    555719 non-null  float64
 13  long                   555719 non-null  float64
 14  city_pop               555719 non-null  i

In [6]:
df['is_fraud'].value_counts()

is_fraud
0    553574
1      2145
Name: count, dtype: int64

# Data Dictionary

* `trans_date_trans_time` - 거래 타임스탬프 (날짜 및 시간)
* `cc_num` - 신용카드 번호
* `merchant` - 가맹점 이름
* `category` - 거래 카테고리
* `amt` - 거래 금액
* `first` - 카드 소지자 이름 (이름)
* `last` - 카드 소지자 이름 (성)
* `gender` - 카드 소지자의 성별
* `street` - 카드 소시자 주소
* `city` - 카드 소지자 도시
* `state` - 카드 소지자 주 (state)
* `zip` - 카드 소지자 우편번호
* `lat` - 카드 소지자 위도
* `long` - 카드 소지자 경도
* `city_pop` - 도시 인구수
* `job` - 카드 소지자의 직업
* `dob` - 카드 소지자의 생년월일
* `trans_num` - 거래 고유 번호
* `unix_time` - 유닉스 시간
* `merch_lat` - 가맹점 위도
* `merch_long` - 가맹점 경도
* `is_fraud` - 사기 여부 (1: 사기, 0: 정상)

데이터 제거

- cc_num
- zip
- trans_num
- first, last
- merchant
- street
- unix_time
- state

In [7]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

In [8]:
# trans_date_trans_time 에서 시간, 분, 요일, 년도, 월, 일 추출 

df['year'] = df['trans_date_trans_time'].dt.year
df['month'] = df['trans_date_trans_time'].dt.month
df['day'] = df['trans_date_trans_time'].dt.day
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek

In [9]:
def season(x):
    if x in [1 ,2 ,3]:
        return 1
    if x in [4 ,5 ,6]:
        return 2
    if x in [7 ,8 ,9]:
        return 3
    if x in [10 ,11 ,12]:
        return 4

df['season'] = df['month'].apply(season)

In [10]:
df.head(2)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,day_of_week,season
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,2020,6,21,12,6,2
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,2020,6,21,12,6,2


In [11]:
# dob에서 나이 추출, age는 거래당시의 나이

df['dob'] = pd.to_datetime(df['dob'])

from dateutil.relativedelta import relativedelta

def calculate_age(born, trans_date):
    return relativedelta(trans_date, born).years

df['age'] = df.apply(lambda x: calculate_age(x['dob'], x['trans_date_trans_time']), axis=1)

In [12]:
df.head(2)

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,day_of_week,season,age
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,2020,6,21,12,6,2,52
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,2020,6,21,12,6,2,30


In [13]:
# 전처리 이후 필요없는 변수(trans_date_trans_time, dob) 제거

df = df.drop(['trans_date_trans_time', 'dob'], axis=1)

In [14]:
df.head(2)

,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,trans_num,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,day_of_week,season,age
0,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0,2020,6,21,12,6,2,52
1,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0,2020,6,21,12,6,2,30


In [15]:
# 이름 및 거래 고유 번호 제거

df_non_name = df.drop(['cc_num', 'first', 'last', 'street', 'zip', 'trans_num'], axis=1) 

In [16]:
df_non_name.head(2)

,merchant,category,amt,gender,city,state,lat,long,city_pop,job,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,day_of_week,season,age
0,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,33.9659,-80.9355,333497,Mechanical engineer,1371816865,33.986391,-81.200714,0,2020,6,21,12,6,2,52
1,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,40.3207,-110.4360,302,"Sales professional, IT",1371816873,39.450498,-109.960431,0,2020,6,21,12,6,2,30


In [17]:
df_non_name.info()

<class 'pandas.core.frame.DataFrame'>
Index: 555719 entries, 0 to 555718
Data columns (total 21 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   merchant     555719 non-null  object 
 1   category     555719 non-null  object 
 2   amt          555719 non-null  float64
 3   gender       555719 non-null  object 
 4   city         555719 non-null  object 
 5   state        555719 non-null  object 
 6   lat          555719 non-null  float64
 7   long         555719 non-null  float64
 8   city_pop     555719 non-null  int64  
 9   job          555719 non-null  object 
 10  unix_time    555719 non-null  int64  
 11  merch_lat    555719 non-null  float64
 12  merch_long   555719 non-null  float64
 13  is_fraud     555719 non-null  int64  
 14  year         555719 non-null  int32  
 15  month        555719 non-null  int32  
 16  day          555719 non-null  int32  
 17  hour         555719 non-null  int32  
 18  day_of_week  555719 non-null 

In [18]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # 지구 반지름 (km 단위)
    
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [19]:
df_non_name['distance_km'] = haversine_distance(df_non_name['lat'], df_non_name['long'], df_non_name['merch_lat'], df_non_name['merch_long'])

In [20]:
df_non_name.head(2)

,merchant,category,amt,gender,city,state,lat,long,city_pop,job,unix_time,merch_lat,merch_long,is_fraud,year,month,day,hour,day_of_week,season,age,distance_km
0,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,33.9659,-80.9355,333497,Mechanical engineer,1371816865,33.986391,-81.200714,0,2020,6,21,12,6,2,52,24.561462
1,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,40.3207,-110.4360,302,"Sales professional, IT",1371816873,39.450498,-109.960431,0,2020,6,21,12,6,2,30,104.925092


In [37]:
df_non_name.nunique()

merchant          693
category           14
amt             37256
gender              2
city              849
state              50
lat               910
long              910
city_pop          835
job               478
unix_time      544760
merch_lat      546490
merch_long     551770
is_fraud            2
year                1
month               7
day                31
hour               24
day_of_week         7
season              3
age                82
distance_km    555719
dtype: int64

In [21]:
df_final = df_non_name.drop(['lat', 'long', 'merch_lat', 'merch_long', 'year'], axis=1) 

In [22]:
df_final.head(2)

,merchant,category,amt,gender,city,state,city_pop,job,unix_time,is_fraud,month,day,hour,day_of_week,season,age,distance_km
0,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,333497,Mechanical engineer,1371816865,0,6,21,12,6,2,52,24.561462
1,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,302,"Sales professional, IT",1371816873,0,6,21,12,6,2,30,104.925092


In [23]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 555719 entries, 0 to 555718
Data columns (total 17 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   merchant     555719 non-null  object 
 1   category     555719 non-null  object 
 2   amt          555719 non-null  float64
 3   gender       555719 non-null  object 
 4   city         555719 non-null  object 
 5   state        555719 non-null  object 
 6   city_pop     555719 non-null  int64  
 7   job          555719 non-null  object 
 8   unix_time    555719 non-null  int64  
 9   is_fraud     555719 non-null  int64  
 10  month        555719 non-null  int32  
 11  day          555719 non-null  int32  
 12  hour         555719 non-null  int32  
 13  day_of_week  555719 non-null  int32  
 14  season       555719 non-null  int64  
 15  age          555719 non-null  int64  
 16  distance_km  555719 non-null  float64
dtypes: float64(2), int32(4), int64(5), object(6)
memory usage: 67.8+ MB


# 로지스틱회귀 - 통계적 검증

In [24]:
from sklearn.preprocessing import StandardScaler

df_final['city_pop'] = df_final['city_pop'].apply(np.log)

num_cols = ['amt', 'city_pop', 'age', 'unix_time', 'distance_km']

scaler = StandardScaler()
df_final[num_cols] = scaler.fit_transform(df_final[num_cols])

In [25]:
def reduce_categories(df, col_list, top_n=30):
    df = df.copy()
    for col in col_list:
        top_cats = df[col].value_counts().nlargest(top_n).index
        df[col] = df[col].apply(lambda x: x if x in top_cats else 'etc')
    return df

In [26]:
cat_cols = ['merchant', 'city', 'job', 'state', 'category', 'gender']
df_final = reduce_categories(df_final, cat_cols, top_n=30)

In [27]:
other_cols = ['month', 'day', 'hour', 'day_of_week', 'season']

In [28]:
target = 'is_fraud'
features = num_cols + cat_cols + other_cols
formula = f"{target} ~ " + " + ".join([
    f"C({col})" if col in cat_cols else col for col in features
])

In [29]:
formula

'is_fraud ~ amt + city_pop + age + unix_time + distance_km + C(merchant) + C(city) + C(job) + C(state) + C(category) + C(gender) + month + day + hour + day_of_week + season'

In [30]:
df_final.head()

,merchant,category,amt,gender,city,state,city_pop,job,unix_time,is_fraud,month,day,hour,day_of_week,season,age,distance_km
0,etc,personal_care,-0.424463,M,etc,SC,1.782476,etc,-1.703871,0,6,21,12,6,2,0.321790,-1.770215
1,etc,personal_care,-0.252337,F,etc,etc,-1.077257,etc,-1.703869,0,6,21,12,6,2,-0.940243,0.989805
2,etc,health_fitness,-0.179353,F,etc,NY,0.856520,"Librarian, public",-1.703865,0,6,21,12,6,2,0.149695,-0.584703
3,etc,misc_pos,-0.059605,M,etc,FL,1.045175,etc,-1.703861,0,6,21,12,6,2,-0.825513,-1.662474
4,etc,travel,-0.422358,M,etc,MI,-0.540162,etc,-1.703861,0,6,21,12,6,2,1.010171,0.969542


In [31]:
df_final.corr(numeric_only=True).abs()

,amt,city_pop,unix_time,is_fraud,month,day,hour,day_of_week,season,age,distance_km
amt,1.000000,0.010262,0.000974,0.182267,0.000717,0.000829,0.029860,0.003414,0.000550,0.012864,0.000767
city_pop,0.010262,1.000000,0.001001,0.000890,0.001491,0.002331,0.030808,0.003284,0.002261,0.147743,0.021175
unix_time,0.000974,0.001001,1.000000,0.013066,0.988955,0.044348,0.000304,0.009987,0.890485,0.006830,0.000419
is_fraud,0.182267,0.000890,0.013066,1.000000,0.011748,0.009203,0.011686,0.009365,0.008153,0.007334,0.000233
month,0.000717,0.001491,0.988955,0.011748,1.000000,0.104058,0.004995,0.004090,0.908028,0.007967,0.000382
day,0.000829,0.002331,0.044348,0.009203,0.104058,1.000000,0.000161,0.038866,0.139583,0.002373,0.000218
hour,0.029860,0.030808,0.000304,0.011686,0.004995,0.000161,1.000000,0.001516,0.006385,0.173695,0.000527
day_of_week,0.003414,0.003284,0.009987,0.009365,0.004090,0.038866,0.001516,1.000000,0.012446,0.004135,0.000930
season,0.000550,0.002261,0.890485,0.008153,0.908028,0.139583,0.006385,0.012446,1.000000,0.007745,0.000352
age,0.012864,0.147743,0.006830,0.007334,0.007967,0.002373,0.173695,0.004135,0.007745,1.000000,0.003165


In [32]:
print(df_final.nunique())

merchant           31
category           14
amt             37256
gender              2
city               31
state              31
city_pop          835
job                31
unix_time      544760
is_fraud            2
month               7
day                31
hour               24
day_of_week         7
season              3
age                82
distance_km    555719
dtype: int64


In [33]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 555719 entries, 0 to 555718
Data columns (total 17 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   merchant     555719 non-null  object 
 1   category     555719 non-null  object 
 2   amt          555719 non-null  float64
 3   gender       555719 non-null  object 
 4   city         555719 non-null  object 
 5   state        555719 non-null  object 
 6   city_pop     555719 non-null  float64
 7   job          555719 non-null  object 
 8   unix_time    555719 non-null  float64
 9   is_fraud     555719 non-null  int64  
 10  month        555719 non-null  int32  
 11  day          555719 non-null  int32  
 12  hour         555719 non-null  int32  
 13  day_of_week  555719 non-null  int32  
 14  season       555719 non-null  int64  
 15  age          555719 non-null  float64
 16  distance_km  555719 non-null  float64
dtypes: float64(5), int32(4), int64(2), object(6)
memory usage: 67.8+ MB


## 다중공산성 진단

In [50]:
# 다중공산성 문제 진단 - VIF(분산 팽창 인수) 확인
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df_final_x = df_final.drop(['is_fraud'], axis = 1)
df_final_x = pd.get_dummies(df_final_x, dtype=int, drop_first=True)

# bool 타입 칼럼 int로 변환
for col in df_final_x.select_dtypes(include='bool').columns:
    df_final_x[col] = df_final_x[col].astype(int)

# 분산이 0인 컬럼 제거
singular_cols = [col for col in df_final_x.columns if df_final_x[col].nunique() == 1]
if len(singular_cols) > 0:
    print(f"\n--- Removing columns with zero variance: {singular_cols} ---")
    df_final_x = df_final_x.drop(columns=singular_cols)

# VIF 계산 시작
print("\n--- VIF (Variance Inflation Factor) ---")
vif_data = pd.DataFrame()
vif_data["feature"] = df_final_x.columns

# 모든 독립 변수에 대해 VIF 계산
vif_values = []
for i in range(df_final_x.shape[1]):
    try:
        vif_values.append(variance_inflation_factor(df_final_x.values, i))
    except Exception as e:
        # 혹시 모를 다른 오류 처리 (예: 여전히 분산 0인 컬럼 등)
        print(f"Error calculating VIF for column {df_final_x.columns[i]}: {e}")
        vif_values.append(np.nan) # 오류 발생 시 NaN으로 표시

vif_data["VIF"] = vif_values
print(vif_data.sort_values(by="VIF", ascending=False))


--- VIF (Variance Inflation Factor) ---
                        feature         VIF
3                         month  894.506134
84                     city_etc  420.321273
7                        season  199.718672
144                     job_etc  169.320379
2                     unix_time   29.767331
..                          ...         ...
37    merchant_fraud_Schumm PLC    1.021171
24   merchant_fraud_Kilback LLC    1.020837
12     merchant_fraud_Boyer PLC    1.020631
27      merchant_fraud_Kuhn LLC    1.020307
9                   distance_km    1.004552

[145 rows x 2 columns]


In [57]:
pd.set_option('display.max_rows', 200)

In [58]:
vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)

,feature,VIF
0,month,894.506134
1,city_etc,420.321273
2,season,199.718672
3,job_etc,169.320379
4,unix_time,29.767331
5,hour,9.552664
6,state_etc,7.423131
7,day,6.659125
8,state_TX,4.181406
9,category_gas_transport,3.825662


In [59]:
pd.reset_option('display.max_rows')

## 범주형 변수 우도비 검정

In [76]:
all_model_features = num_cols + cat_cols + other_cols

In [74]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 가중치 지정
df_final['weights'] = df_final['is_fraud'].apply(lambda x: 258 if x == 1 else 1)

full_model = smf.glm(formula=formula, data=df_final, family=sm.families.Binomial())
full_result = full_model.fit(weights=df_final['weights'])
full_result.summary().tables[0] # 첫 번째 요약 테이블만 출력하여 간결하게

Dep. Variable:,is_fraud,No. Observations:,555719
Model:,GLM,Df Residuals:,555574
Model Family:,Binomial,Df Model:,144
Link Function:,Logit,Scale:,1.0000
Method:,IRLS,Log-Likelihood:,-11083.
Date:,"Mon, 30 Jun 2025",Deviance:,22166.
Time:,00:43:49,Pearson chi2:,2.51e+15
No. Iterations:,30,Pseudo R-squ. (CS):,0.01066
Covariance Type:,nonrobust,,


In [78]:
from scipy import stats

# 각 범주형 변수에 대한 우도비 검정 (Likelihood Ratio Test)
categorical_features_to_test = ['merchant', 'city', 'job', 'state', 'category', 'gender']

for feature in categorical_features_to_test:
    print(f"\n[검정 중]: '{feature}' 변수")

    # 현재 범주형 변수를 제외한 축소 모델(Reduced Model)의 포뮬러 구성
    reduced_features_for_formula = [f for f in all_model_features if f != feature] # C() 래핑 전의 feature 이름으로 제거
    reduced_formula = f"{target} ~ " + " + ".join([
        f"C({col})" if col in cat_cols else col # 원래 제공된 코드처럼 cat_cols만 C()
        for col in reduced_features_for_formula
    ])

    # 축소 모델 학습 (동일한 weights 적용)
    try:
        reduced_model = smf.glm(formula=reduced_formula, data=df_final, family=sm.families.Binomial())
        reduced_result = reduced_model.fit(weights=df_final['weights'])

        # LRT 통계량 계산
        lrt_statistic = 2 * (full_result.llf - reduced_result.llf)

        # 자유도(Degrees of Freedom) 차이 계산
        df_diff = len(full_result.params) - len(reduced_result.params)

        # p-value 계산
        lrt_pvalue = stats.chi2.sf(lrt_statistic, df_diff)

        print(f"  - LRT 통계량: {lrt_statistic:.4f}")
        print(f"  - 자유도 (df): {df_diff}")
        print(f"  - LRT p-value: {lrt_pvalue:.6f}")

        if lrt_pvalue < 0.05:
            print(f"  결론: '{feature}' 변수는 통계적으로 유의미합니다 (p < 0.05). 모델에 유의미한 기여를 합니다.")
        else:
            print(f"  결론: '{feature}' 변수는 통계적으로 유의미하지 않습니다 (p >= 0.05). 모델에서 제거하는 것을 고려할 수 있습니다.")
    except Exception as e:
        print(f"  오류 발생: '{feature}' 변수에 대한 축소 모델 학습 중 문제가 발생했습니다: {e}")
        print("  (이 오류는 데이터의 완전 분리, 작은 클래스 수, 또는 기타 수렴 문제로 인해 발생할 수 있습니다. 특히 'etc' 범주가 생긴 후 해당 범주만으로 클래스가 분리될 경우 발생하기 쉽습니다.)")
        print("  해결책: 범주 통합 시 top_n 값 조정, 또는 분리 문제를 일으키는 데이터 포인트 확인, 또는 sklearn.linear_model.LogisticRegression과 같은 정규화 모델 고려.")


[검정 중]: 'merchant' 변수
  - LRT 통계량: 26.4509
  - 자유도 (df): 30
  - LRT p-value: 0.651932
  결론: 'merchant' 변수는 통계적으로 유의미하지 않습니다 (p >= 0.05). 모델에서 제거하는 것을 고려할 수 있습니다.

[검정 중]: 'city' 변수
  - LRT 통계량: 417.6055
  - 자유도 (df): 30
  - LRT p-value: 0.000000
  결론: 'city' 변수는 통계적으로 유의미합니다 (p < 0.05). 모델에 유의미한 기여를 합니다.

[검정 중]: 'job' 변수
  - LRT 통계량: 384.7780
  - 자유도 (df): 30
  - LRT p-value: 0.000000
  결론: 'job' 변수는 통계적으로 유의미합니다 (p < 0.05). 모델에 유의미한 기여를 합니다.

[검정 중]: 'state' 변수
  - LRT 통계량: 244.5449
  - 자유도 (df): 30
  - LRT p-value: 0.000000
  결론: 'state' 변수는 통계적으로 유의미합니다 (p < 0.05). 모델에 유의미한 기여를 합니다.

[검정 중]: 'category' 변수
  - LRT 통계량: 2491.9353
  - 자유도 (df): 13
  - LRT p-value: 0.000000
  결론: 'category' 변수는 통계적으로 유의미합니다 (p < 0.05). 모델에 유의미한 기여를 합니다.

[검정 중]: 'gender' 변수
  오류 발생: 'gender' 변수에 대한 축소 모델 학습 중 문제가 발생했습니다: 
  (이 오류는 데이터의 완전 분리, 작은 클래스 수, 또는 기타 수렴 문제로 인해 발생할 수 있습니다. 특히 'etc' 범주가 생긴 후 해당 범주만으로 클래스가 분리될 경우 발생하기 쉽습니다.)
  해결책: 범주 통합 시 top_n 값 조정, 또는 분리 문제를 일으키는 데이터 포인트 확인, 또는 sklearn.linear_

## pvalue와 오즈비 확인

In [34]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 가중치 지정
df_final['weights'] = df_final['is_fraud'].apply(lambda x: 258 if x == 1 else 1)

# 모델 생성
model = smf.glm(formula=formula, data=df_final, 
                family=sm.families.Binomial())

# 학습 (weights 적용)
result = model.fit(weights=df_final['weights'])

In [35]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               is_fraud   No. Observations:               555719
Model:                            GLM   Df Residuals:                   555574
Model Family:                Binomial   Df Model:                          144
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -11083.
Date:                Sun, 29 Jun 2025   Deviance:                       22166.
Time:                        21:58:41   Pearson chi2:                 2.51e+15
No. Iterations:                    30   Pseudo R-squ. (CS):            0.01066
Covariance Type:            nonrobust                                         
===============================================================================================================================
                                                                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------------
Intercept                                                     283.5971     28.646      9.900      0.000     227.451     339.743
C(merchant)[T.fraud_Bins-Rice]                                  0.2038      0.513      0.397      0.691      -0.802       1.209
C(merchant)[T.fraud_Boyer PLC]                                  0.2419      0.290      0.834      0.404      -0.327       0.810
C(merchant)[T.fraud_Brekke and Sons]                           -0.5572      0.716     -0.778      0.437      -1.961       0.847
C(merchant)[T.fraud_Cormier LLC]                               -0.6582      0.461     -1.428      0.153      -1.562       0.245
C(merchant)[T.fraud_Corwin-Collins]                            -0.5881      0.716     -0.821      0.412      -1.992       0.816
C(merchant)[T.fraud_Dickinson Ltd]                             -1.0905      1.005     -1.086      0.278      -3.059       0.878
C(merchant)[T.fraud_Eichmann, Bogan and Rodriguez]             -0.5207      0.716     -0.727      0.467      -1.925       0.883
C(merchant)[T.fraud_Emard Inc]                                 -0.6136      0.716     -0.856      0.392      -2.018       0.791
C(merchant)[T.fraud_Erdman-Kertzmann]                          -0.5619      0.716     -0.784      0.433      -1.966       0.842
C(merchant)[T.fraud_Friesen-D'Amore]                           -0.5715      0.716     -0.798      0.425      -1.976       0.833
C(merchant)[T.fraud_Greenholt, Jacobi and Gleason]             -0.1417      0.589     -0.241      0.810      -1.296       1.012
C(merchant)[T.fraud_Harris Inc]                                -1.2836      1.007     -1.275      0.202      -3.257       0.689
C(merchant)[T.fraud_Huels-Hahn]                                -0.5379      0.716     -0.751      0.453      -1.942       0.866
C(merchant)[T.fraud_Kilback LLC]                                0.3562      0.266      1.341      0.180      -0.164       0.877
C(merchant)[T.fraud_Kling Inc]                                 -0.1503      0.589     -0.255      0.799      -1.304       1.004
C(merchant)[T.fraud_Koss and Sons]                             -0.1281      0.589     -0.217      0.828      -1.282       1.026
C(merchant)[T.fraud_Kuhn LLC]                                   0.2839      0.321      0.884      0.377      -0.346       0.914
C(merchant)[T.fraud_Ledner-Pfannerstill]                       -0.1268      0.589     -0.215      0.829      -1.281       1.027
C(merchant)[T.fraud_Mraz-Herzog]                                0.1800      0.513      0.351      0.726      -0.826       1.186
C(merchant)[T.fraud_Parisian and Sons]                          0.8991      0.353      2.549      0.011       0.208       1.590
C(merchant)

In [60]:
significant_pvalues = result.pvalues[result.pvalues < 0.05]

In [61]:
significant_pvalues

Intercept                                   4.167300e-23
C(merchant)[T.fraud_Parisian and Sons]      1.080811e-02
C(city)[T.Andrews]                          4.622186e-02
C(city)[T.Birmingham]                       2.443267e-02
C(city)[T.Meridian]                         1.695549e-02
C(city)[T.etc]                              3.733447e-02
C(job)[T.Clothing/textile technologist]     7.293225e-03
C(job)[T.Comptroller]                       2.009252e-02
C(job)[T.Sub]                               1.438807e-04
C(job)[T.Systems developer]                 2.941776e-03
C(state)[T.AR]                              2.258712e-02
C(state)[T.CA]                              2.366766e-03
C(state)[T.LA]                              4.076274e-03
C(state)[T.MI]                              2.352856e-02
C(state)[T.NE]                              3.243392e-02
C(state)[T.OH]                              3.876374e-04
C(state)[T.PA]                              1.060214e-02
C(state)[T.TX]                 

In [66]:
significant_features = significant_pvalues.index
significant_coefficients = result.params[significant_features]
odds_ratios = np.exp(significant_coefficients)

In [69]:
odds_ratios

Intercept                                  1.461035e+123
C(merchant)[T.fraud_Parisian and Sons]      2.457269e+00
C(city)[T.Andrews]                          3.198288e-01
C(city)[T.Birmingham]                       2.576889e+00
C(city)[T.Meridian]                         2.596826e-01
C(city)[T.etc]                              5.012106e-01
C(job)[T.Clothing/textile technologist]     3.084912e+00
C(job)[T.Comptroller]                       2.488578e+00
C(job)[T.Sub]                               4.777693e+00
C(job)[T.Systems developer]                 3.173941e+00
C(state)[T.AR]                              5.967894e-01
C(state)[T.CA]                              5.716758e-01
C(state)[T.LA]                              4.671298e-01
C(state)[T.MI]                              6.374670e-01
C(state)[T.NE]                              6.193944e-01
C(state)[T.OH]                              4.660325e-01
C(state)[T.PA]                              6.379852e-01
C(state)[T.TX]                 